<a href="https://colab.research.google.com/github/tregrwells/timing-law/blob/main/Timing_Law_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# Timing Law Analysis - Wells (2026)
# ============================================================================
# This notebook reproduces all analyses from:
# "The Inverse Relationship Between Target Duration and Subjective Time Dilation:
# A Large-Scale Behavioral Study"
#
# Data: Timing Database (OSF.io/vrwjz)
# Data hosted on GitHub: github.com/tregrwells/timing-law
# Author: Treg R. Wells
# ============================================================================

# ----------------------------------------------------------------------------
# 1. Setup
# ----------------------------------------------------------------------------
!pip install pandas numpy matplotlib scipy seaborn scikit-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

# ----------------------------------------------------------------------------
# 2. Load Data from GitHub (or fallback to manual upload)
# ----------------------------------------------------------------------------
# Corrected URL with the correct username and filename
GITHUB_RAW_URL = "https://raw.githubusercontent.com/tregrwells/timing-law/main/TR_dataset-2026-06-29.csv"

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print(f"✅ Loaded {len(df)} rows from GitHub.")
except Exception as e:
    print(f"❌ Could not load from GitHub: {e}")
    print("Please upload the CSV file manually.")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    print(f"✅ Loaded {len(df)} rows from manual upload.")

target_col = 'Target_Duration'
resp_col = 'Response'

# ----------------------------------------------------------------------------
# 3. Data Cleaning
# ----------------------------------------------------------------------------
df['tau'] = df[resp_col] / df[target_col]
df = df[(df['tau'] > 0.1) & (df['tau'] < 5)]
print(f"After cleaning: {len(df)} valid trials")

# ----------------------------------------------------------------------------
# 4. Figure 1: Subject-Level Correlations
# ----------------------------------------------------------------------------
subjects = df['Subj_idx'].unique()
correlations = []

for subj in subjects:
    subj_df = df[df['Subj_idx'] == subj]
    if len(subj_df) < 10:
        continue
    grouped = subj_df.groupby(target_col).agg(tau_mean=('tau', 'mean')).reset_index()
    if len(grouped) < 3:
        continue
    corr, _ = pearsonr(grouped[target_col], grouped['tau_mean'])
    correlations.append(corr)

n_neg = sum(1 for c in correlations if c < 0)
pct_neg = 100 * n_neg / len(correlations)
print(f"N = {len(correlations)} subjects, {pct_neg:.1f}% negative correlation")

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.hist(correlations, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='r = 0')
ax1.text(0.02, 0.95, f'{pct_neg:.1f}% negative\n(N={len(correlations)})',
         transform=ax1.transAxes, fontsize=14,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax1.set_xlabel('Correlation (target duration vs. τ)')
ax1.set_ylabel('Number of Subjects')
ax1.set_title('Figure 1: Subject-Level Correlations')
ax1.legend()
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figure1_final.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------------------
# 5. Figure 2: Group-Level Sigmoid Fit (with fixed label)
# ----------------------------------------------------------------------------
grouped = df.groupby(target_col).agg(
    tau_mean=('tau', 'mean'),
    tau_sem=('tau', 'sem')
).reset_index()
grouped = grouped.sort_values(target_col)

def sigmoid(x, a_crit, delta_a, eps0):
    return eps0 + (1 - eps0) / (1 + np.exp(-(x - a_crit) / delta_a))

durations = grouped[target_col].values
a_vals = 1.0 - (durations - durations.min()) / (durations.max() - durations.min())
eps_vals = 1.0 / grouped['tau_mean'].values
eps_sem = grouped['tau_sem'].values / (grouped['tau_mean'].values ** 2)

popt, _ = curve_fit(sigmoid, a_vals, eps_vals, p0=[0.5, 0.1, 0.7],
                    bounds=([0.1, 0.01, 0.1], [0.9, 0.5, 1.5]))
a_crit, delta_a, eps0 = popt

fig2, ax2 = plt.subplots(figsize=(10, 6))
ax2.errorbar(a_vals, eps_vals, yerr=eps_sem, fmt='o', color='steelblue',
             capsize=3, label='Data (ε = 1/τ)')
a_cont = np.linspace(0, 1, 200)
eps_cont = sigmoid(a_cont, a_crit, delta_a, eps0)
# Use raw string to avoid escape sequence warnings
ax2.plot(a_cont, eps_cont, 'r-', linewidth=2.5,
         label=rf'Fit: $a_{{crit}}$={a_crit:.3f}, $\epsilon_0$={eps0:.3f}')
ax2.axvline(x=a_crit, color='gray', linestyle='--', alpha=0.7)
ax2.set_xlabel('Scale Parameter (a) → Descriptive Coordinate')
ax2.set_ylabel('ε = 1/τ')
ax2.set_title('Figure 2: Group-Level Sigmoid Fit')
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figure2_final.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------------------
# 6. Model Comparison
# ----------------------------------------------------------------------------
def linear(x, m, b): return m * x + b
def quadratic(x, a, b, c): return a * x**2 + b * x + c
def power_law(x, alpha, beta, gamma): return alpha * (x ** beta) + gamma

models = {
    'Linear': (linear, [0.5, 0.5]),
    'Quadratic': (quadratic, [0.5, 0.5, 0.5]),
    'Power': (power_law, [0.5, 0.5, 0.5]),
    'Sigmoid': (sigmoid, [0.5, 0.1, 0.7])
}

results = {}
for name, (func, p0) in models.items():
    try:
        popt, _ = curve_fit(func, a_vals, eps_vals, p0=p0)
        y_pred = func(a_vals, *popt)
        resid = eps_vals - y_pred
        rss = np.sum(resid**2)
        n = len(eps_vals)
        k = len(popt)
        r2 = 1 - rss / np.sum((eps_vals - np.mean(eps_vals))**2)
        aic = n * np.log(rss/n) + 2*k
        bic = n * np.log(rss/n) + k * np.log(n)
        mse = np.mean(resid**2)
        results[name] = {'R2': r2, 'AIC': aic, 'BIC': bic, 'MSE': mse}
    except:
        print(f"⚠️ {name} fit failed")

# Find best AIC
best_aic = min(results[m]['AIC'] for m in results)
for name, res in results.items():
    res['ΔAIC'] = res['AIC'] - best_aic

# Print table
print("\nMODEL COMPARISON")
print("-" * 70)
print(f"{'Model':<10} {'R²':>8} {'AIC':>10} {'BIC':>10} {'MSE':>10} {'ΔAIC':>10}")
print("-" * 70)
for name, res in results.items():
    print(f"{name:<10} {res['R2']:>8.3f} {res['AIC']:>10.1f} {res['BIC']:>10.1f} {res['MSE']:>10.3f} {res['ΔAIC']:>10.1f}")

# ----------------------------------------------------------------------------
# 7. Figure 3: Individual-Level R² Distribution
# ----------------------------------------------------------------------------
def fit_subject(subj_df):
    grouped = subj_df.groupby(target_col).agg(tau_mean=('tau', 'mean')).reset_index()
    if len(grouped) < 3:
        return None
    durs = grouped[target_col].values
    a_subj = 1.0 - (durs - durs.min()) / (durs.max() - durs.min())
    eps_subj = 1.0 / grouped['tau_mean'].values
    try:
        popt, _ = curve_fit(sigmoid, a_subj, eps_subj, p0=[0.5, 0.1, 0.7],
                            bounds=([0.1, 0.01, 0.1], [0.9, 0.5, 1.5]))
        y_pred = sigmoid(a_subj, *popt)
        r2 = 1 - np.sum((eps_subj - y_pred)**2) / np.sum((eps_subj - np.mean(eps_subj))**2)
        return r2
    except:
        return None

ind_r2 = []
for subj in subjects:
    subj_df = df[df['Subj_idx'] == subj]
    if len(subj_df) < 10:
        continue
    r2 = fit_subject(subj_df)
    if r2 is not None:
        ind_r2.append(r2)

fig3, ax3 = plt.subplots(figsize=(8, 4))
ax3.hist(ind_r2, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax3.axvline(0.5, color='red', linestyle='--', label='R² = 0.5')
ax3.set_xlabel('R²')
ax3.set_ylabel('Number of Subjects')
ax3.set_title('Figure 3: Individual-Level Fit Quality')
ax3.legend()
ax3.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('individual_r2.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------------------
# 8. Cross-Validation
# ----------------------------------------------------------------------------
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_grouped = train_df.groupby(target_col).agg(tau_mean=('tau', 'mean')).reset_index()
train_grouped = train_grouped.sort_values(target_col)
train_durs = train_grouped[target_col].values
train_a = 1.0 - (train_durs - train_durs.min()) / (train_durs.max() - train_durs.min())
train_eps = 1.0 / train_grouped['tau_mean'].values

popt_train, _ = curve_fit(sigmoid, train_a, train_eps, p0=[0.5, 0.1, 0.7],
                          bounds=([0.1, 0.01, 0.1], [0.9, 0.5, 1.5]))

test_grouped = test_df.groupby(target_col).agg(tau_mean=('tau', 'mean')).reset_index()
test_grouped = test_grouped.sort_values(target_col)
test_durs = test_grouped[target_col].values
test_a = 1.0 - (test_durs - test_durs.min()) / (test_durs.max() - test_durs.min())
test_eps = 1.0 / test_grouped['tau_mean'].values
test_pred = sigmoid(test_a, *popt_train)
test_r2 = r2_score(test_eps, test_pred)
print(f"\nCross-Validation Test R² = {test_r2:.4f}")

# ----------------------------------------------------------------------------
# 9. Done
# ----------------------------------------------------------------------------
print("\n✅ All analyses complete.")
print("Figures saved: figure1_final.pdf, figure2_final.pdf, individual_r2.pdf")